In [2]:
from flodr import FloDR, viz
import numpy as np
import time
import scanpy as sc
from sklearn.decomposition import PCA


In [3]:
import scripts.datasets as ds
data = ds.get_paul15()
X,y = data[0], data[1]
difference = lambda x,y: np.sum(np.abs(x-y))

100%|██████████| 9.82M/9.82M [00:00<00:00, 68.5MB/s]


In [4]:
# load data + generalized normalization
sc.settings.verbosity = 0
adata = sc.datasets.paul15()
labels = adata.obs['paul15_clusters'].to_numpy()
sc.pp.filter_genes(adata, min_counts=1)
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

# split data into train/test and select highly variable genes
N, SEED = int(0.8 * len(adata.X)), 3
rng = np.random.default_rng(SEED)
perm = rng.permutation(len(adata.X))
test_idx, train_idx = perm[N: ], perm[:N]

train_view = adata[train_idx, :]
sc.pp.highly_variable_genes(train_view, n_top_genes=1000)
hvg_mask = train_view.var.highly_variable

train = adata[train_idx, hvg_mask].copy()
train_org = train.copy()
test = adata[test_idx, hvg_mask].copy()
ground_truth = np.asarray(adata[:, hvg_mask].X.copy(), np.float64)


In [5]:
from sklearn.decomposition import PCA
from anndata import AnnData
def fit_preprocessor(train_adata:AnnData):
    """Fit scaling, PCA, and component standardization on training data."""
    # Scale (modifies train_adata in place)
    sc.pp.scale(train_adata, max_value=10)
    gene_mean = train_adata.var['mean'].values.astype(np.float64)
    gene_std  = train_adata.var['std'].values.astype(np.float64) 
    
    train_x = np.asarray(train_adata.X, np.float64)   # scaled
    
    # PCA
    pca = PCA(n_components=50, svd_solver="randomized", random_state=0)
    pcs_train = pca.fit_transform(train_x)
    
    # Component-wise standardization
    pc_mean = pcs_train.mean(axis=0)
    pc_std  = pcs_train.std(axis=0) + 1e-9
    pcs_train_std = (pcs_train - pc_mean) / pc_std
    
    params = {
        'gene_mean': gene_mean,
        'gene_std': gene_std,
        'pca': pca,
        'pc_mean': pc_mean,
        'pc_std': pc_std
    }
    return pcs_train_std, params

def apply_forward(X_log:np.ndarray, params):
    """Apply scaling, PCA, and component standardization to log‑normalised HVG data."""
    X_scaled = (X_log - params['gene_mean']) / params['gene_std']
    X_scaled = np.clip(X_scaled, None, 10)
    pcs = params['pca'].transform(X_scaled)
    return (pcs - params['pc_mean']) / params['pc_std']

def apply_inverse(Z_std:np.ndarray, params):
    """Map from standardized PCA-50 space back to log‑normalised HVG gene expression."""
    Z_raw = Z_std * params['pc_std'] + params['pc_mean']
    X_scaled = params['pca'].inverse_transform(Z_raw)
    return X_scaled * params['gene_std'] + params['gene_mean']

In [ ]:
print(f"train {len(train_idx)} test {len(test_idx)}")

t0 = time.perf_counter()
model = FloDR(w=2.0, random_state=SEED, density=True)
train_x, params =  fit_preprocessor(train_org.copy())
train_z = model.fit(train_x)
Y = model.embedding_

print(f"fit {time.perf_counter() - t0:.0f}s")
print(f"embedding {Y.shape}")
print(f"latent round trip, max abs error: {model.roundtrip_:.2e}")

In [ ]:
difference(model.inverse_transform(Y), train_x)


np.float64(130510.3261396563)

In [ ]:
test_standardized = apply_forward(test.X.copy(), params)

test_emb = model.transform(test_standardized)
x_pred = model.inverse_transform(test_emb)

# get these from params, or just replace with apply_inverse, delayed until inverse_transform is fixed
xp = (x_pred * train_std) + train_mean 
xp = pca.inverse_transform(xp)
xp = (xp * gene_std) + gene_mean #now in log1p space
xp = np.clip(xp, None, 10) # do we even need to clip here?

ground_truth_test = ground_truth[test_idx]

print(test_emb.shape, x_pred.shape, xp.shape, ground_truth_test.shape)

(546, 2) (546, 50) (546, 1000) (546, 1000)


In [ ]:
from sklearn.metrics import r2_score

x2= train_x
x1= model.inverse_transform(Y)

r2_per_gene = []
for j in range(x1.shape[1]):
    r2 = r2_score(x2[:, j], x1[:, j])
    r2_per_gene.append(r2)

mean_r2 = np.mean(r2_per_gene)
print(f"Mean R² over {len(r2_per_gene)} genes: {mean_r2:.4f}")

Mean R² over 50 genes: -2.3051


In [ ]:
print("x1 mean:", x1.mean(), " std:", x1.std())
print("x2 mean:", x2.mean(), " std:", x2.std())
print("Test embedding (first 5 cells):", test_emb[:5])   # 2D coordinates

x1 mean: 0.3944146034035349  std: 0.7008356022829758
x2 mean: 0.43575634958062853  std: 0.9618141453361703
Test embedding (first 5 cells): [[ 3.5893793  -0.63151515]
 [-3.0865388   5.044631  ]
 [ 6.2961097   1.3488374 ]
 [-2.7886786   1.146043  ]
 [ 0.26250148  2.0070927 ]]


In [7]:
# load data + generalized normalization
sc.settings.verbosity = 0
adata = sc.datasets.paul15()
labels = adata.obs['paul15_clusters'].to_numpy()
sc.pp.filter_genes(adata, min_counts=1)
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
sc.pp.highly_variable_genes(adata, n_top_genes=1000)

# split data into train/test and select highly variable genes
N, SEED = int(0.8 * len(adata.X)), 3
rng = np.random.default_rng(SEED)
perm = rng.permutation(len(adata.X))
test_idx, train_idx = perm[N: ], perm[:N]

train = adata[train_idx, ].copy()
test = adata[test_idx, ].copy()
ground_truth = np.asarray(adata.X.copy(), np.float64)


In [ ]:
print(f"train {len(train_idx)} test {len(test_idx)}")

t0 = time.perf_counter()
model = FloDR(w=2.0, random_state=SEED, density=True, n_components=40)
# train_x, params =  fit_preprocessor(train.copy())
train_x = X[train_idx,:]
train_z = model.fit()
Y = model.embedding_

print(f"fit {time.perf_counter() - t0:.0f}s")
print(f"embedding {Y.shape}")
print(f"latent round trip, max abs error: {model.roundtrip_:.2e}")

train 2184 test 546


KeyboardInterrupt: 

In [15]:
difference(model.inverse_transform(Y), train_x)


ValueError: operands could not be broadcast together with shapes (2730,50) (2184,50) 